#### **The section covers the RAG implementation using PDF Q&A**

In [67]:
import json
# from llama_index.llms.gemini import Gemini
# from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
import nest_asyncio
nest_asyncio.apply()

In [69]:
# !pip install nest_asyncio

In [57]:
# !pip install llama-index-llms-google-genai llama-index-embeddings-google-genai

#### **Before calling any model lets check the model availability**

In [58]:
# open the json file and read the API key
with open("config.json") as f:
    API_KEY=json.load(f)["API_KEY"]

In [59]:
import google.generativeai as genai

genai.configure(api_key=API_KEY)

for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

In [60]:
# Filter only models that support text generation
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [61]:
for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(f"Name: {model.name}")
        print(f"Display name: {model.display_name}")
        print(f"Description: {model.description}")
        print("---")


Name: models/gemini-embedding-001
Display name: Gemini Embedding 001
Description: Obtain a distributed representation of a text.
---
Name: models/gemini-embedding-2-preview
Display name: Gemini Embedding 2 Preview
Description: Obtain a distributed representation of multimodal content.
---


In [62]:
# 1. Setup Gemini
Settings.llm = GoogleGenAI(model="models/gemini-2.5-flash", api_key=API_KEY)

Settings.embed_model = GoogleGenAIEmbedding(model="models/gemini-embedding-001", api_key=API_KEY)


2026-03-13 22:38:41,581 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash "HTTP/1.1 200 OK"


In [63]:
# 2. Load PDFs
documents = SimpleDirectoryReader("./pdfs").load_data()

In [64]:
documents

[Document(id_='2f8e8ebc-b757-4570-a871-aaff8a98f3b8', embedding=None, metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF Q&A RAG\\pdfs\\monopoly_instructions.pdf', 'file_type': 'application/pdf', 'file_size': 625715, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play money and a Banker\'s tray. \nNow there\'s a faster way to play MONOPO

In [65]:
# 3. Create Index (Vector Store)
index = VectorStoreIndex.from_documents(documents)

2026-03-13 22:38:54,653 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"


In [70]:
# 4. Query
query_engine = index.as_query_engine()
response = query_engine.query("Summarize the key findings in the PDF.")
print(response)


2026-03-13 22:42:37,671 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-13 22:42:37,700 - INFO - AFC is enabled with max remote calls: 10.
2026-03-13 22:42:40,805 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


The document describes the MONOPOLY property trading game, suitable for ages 8 and up, and playable by 2 to 8 players. The game includes a board, three dice, tokens, houses, hotels, Chance and Community Chest cards, Title Deed cards, play money, and a Banker's tray.

Players can choose to play by classic rules or use the Speed Die for a faster game. If using the Speed Die, each player receives an extra $1,000 at the start. The Speed Die is not used until a player lands on or passes GO for the first time. Once activated, it is rolled along with the two white dice, and its result (1, 2, or 3) is added to the total of the white dice, allowing for faster movement.

Specific rules for certain board spaces are also detailed. For "Income Tax," players have two options: pay a flat $200 or 10% of their total worth (cash, property values, and building costs). This decision must be made before calculating total worth.

Players can land in "Jail" by landing on the "Go to Jail" space, drawing a "Go

In [71]:
# writing the response to json file
with open("response.json", "w") as f:
    json.dump(str(response), f)